# Day 5: Evaluation

Welcome to Day 5 of the AI agents crash course.

Yesterday you added **tools** and an **agent**. Today: **is it any good?**

You will:

- **Log** interactions to JSON (prototype — not production)
- Use an **LLM as a judge** with **structured output** (Pydantic models)
- **Generate** synthetic test questions
- **Aggregate** pass/fail metrics with **Pandas**

Uses the **DataTalks FAQ** + **text search** (same thread as Days 3–4). Same ideas apply to docs / vector search.

**Needs:** `OPENAI_API_KEY` (e.g. `ai_hero/.env`).


## Setup

```bash
cd course
uv add pandas tqdm requests python-frontmatter minsearch openai python-dotenv pydantic-ai
uv add --dev jupyter
```

This notebook uses **`run_agent_sync`** (thread + `asyncio.run`) so it runs under Jupyter without the event-loop error.


In [1]:
import asyncio
import json
import os
import random
import secrets
import threading
from datetime import datetime
from pathlib import Path

import pandas as pd
import requests
import frontmatter
import io
import zipfile
from tqdm.auto import tqdm
from pydantic import BaseModel
from pydantic_ai import Agent
from pydantic_ai.messages import ModelMessagesTypeAdapter
from dotenv import load_dotenv
from minsearch import Index

# Load .env from ai_hero root
_p = Path.cwd()
for _ in range(10):
    if (_p / ".env").is_file():
        load_dotenv(_p / ".env")
        break
    if _p.parent == _p:
        break
    _p = _p.parent
if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("Set OPENAI_API_KEY")

LOG_DIR = Path("logs")
LOG_DIR.mkdir(exist_ok=True)


def run_agent_sync(agent, user_prompt: str):
    out = {}

    def _go():
        out["r"] = asyncio.run(agent.run(user_prompt))

    t = threading.Thread(target=_go)
    t.start()
    t.join()
    return out["r"]


In [2]:
def read_repo_data(repo_owner, repo_name):
    url = f"https://codeload.github.com/{repo_owner}/{repo_name}/zip/refs/heads/main"
    r = requests.get(url)
    r.raise_for_status()
    out = []
    zf = zipfile.ZipFile(io.BytesIO(r.content))
    for fi in zf.infolist():
        fn = fi.filename
        fl = fn.lower()
        if not (fl.endswith(".md") or fl.endswith(".mdx")):
            continue
        try:
            with zf.open(fi) as f:
                c = f.read().decode("utf-8", errors="ignore")
                post = frontmatter.loads(c)
                d = post.to_dict()
                d["filename"] = fn
                out.append(d)
        except Exception as e:
            print("skip", fn, e)
    zf.close()
    return out


dtc_faq = read_repo_data("DataTalksClub", "faq")
de_dtc_faq = [
    d for d in dtc_faq if "data-engineering" in (d.get("filename") or "").lower()
]
faq_index = Index(text_fields=["question", "content"], keyword_fields=[])
faq_index.fit(de_dtc_faq)
len(de_dtc_faq)


498

In [3]:
def text_search(query: str):
    return faq_index.search(query, num_results=5)


system_prompt_v1 = """
You are a helpful assistant for a course.

Use the search tool to find relevant information from the course materials before answering questions.

If you can find specific information through search, use it to provide accurate answers.
If the search doesn't return relevant results, let the user know and provide general guidance.
""".strip()

agent = Agent(
    "openai:gpt-4o-mini",
    name="faq_agent",
    instructions=system_prompt_v1,
    tools=[text_search],
)


## Logging

**Vibe checks** are fine to start; for real decisions, **record** interactions and score them systematically.

We write one JSON file per run under **`logs/`** (add `logs/` to `.gitignore` in real projects).


In [4]:
def _instructions_text(agent) -> str:
    parts = getattr(agent, "_instructions", None) or []
    return "\n\n".join(str(p) for p in parts)


def log_entry(agent, messages, source="user"):
    fts = getattr(agent, "_function_toolset", None)
    tool_names = list(fts.tools.keys()) if fts is not None else []
    dict_messages = ModelMessagesTypeAdapter.dump_python(messages)
    return {
        "agent_name": agent.name,
        "system_prompt": _instructions_text(agent),
        "model": str(getattr(agent, "_model", "")),
        "tools": tool_names,
        "messages": dict_messages,
        "source": source,
    }


def serializer(obj):
    if isinstance(obj, datetime):
        return obj.isoformat()
    raise TypeError(f"Type {type(obj)} not serializable")


def log_interaction_to_file(agent, messages, source="user"):
    entry = log_entry(agent, messages, source)
    ts_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    rand_hex = secrets.token_hex(3)
    filename = f"{agent.name}_{ts_str}_{rand_hex}.json"
    filepath = LOG_DIR / filename
    with filepath.open("w", encoding="utf-8") as f_out:
        json.dump(entry, f_out, indent=2, default=serializer)
    return filepath


In [5]:
# Single turn: try a question and log `new_messages()`
question = "how do I install Kafka in Python?"
result = run_agent_sync(agent, question)
print(result.output)
path = log_interaction_to_file(agent, result.new_messages(), source="user")
path


To install Kafka in Python, you typically need the `confluent-kafka` library or the `kafka-python` library. Here are the steps for both methods:

### Using `confluent-kafka`:
1. Install `confluent-kafka` using pip:
   ```bash
   pip install confluent-kafka
   ```
   Alternatively, if you are using conda, you can install it with:
   ```bash
   conda install conda-forge::python-confluent-kafka
   ```

2. Also, you may want to install `fastavro` if your application requires it:
   ```bash
   pip install fastavro
   ```

### Using `kafka-python`:
1. First, you might want to uninstall any existing version of `kafka-python` if you encounter issues:
   ```bash
   pip uninstall kafka-python
   ```

2. Then install a specific version of `kafka-python`:
   ```bash
   pip install kafka-python==1.4.6
   ```

3. As an alternative to `kafka-python`, you can use `kafka-python-ng`:
   ```bash
   pip install kafka-python-ng
   ```

These steps should help you set up Kafka with Python. Make sure to choo

WindowsPath('logs/faq_agent_20260329_000550_ab7a02.json')

### References in answers (prompt v2)

Ask the model to cite **FAQ filenames** as GitHub links (DataTalksClub/faq).


In [6]:
system_prompt_v2 = """
You are a helpful assistant for a course.

Use the search tool to find relevant information from the course materials before answering questions.

If you can find specific information through search, use it to provide accurate answers.

Always include references by citing the filename of the source material you used.
When citing, replace "faq-main" in paths with the repo base:
https://github.com/DataTalksClub/faq/blob/main/
Format: [LINK TITLE](FULL_GITHUB_LINK)

If the search doesn't return relevant results, let the user know and provide general guidance.
""".strip()

faq_agent_v2 = Agent(
    "openai:gpt-4o-mini",
    name="faq_agent_v2",
    instructions=system_prompt_v2,
    tools=[text_search],
)

q2 = "can I join late and get a certificate?"
r2 = run_agent_sync(faq_agent_v2, q2)
print(r2.output)
log_interaction_to_file(faq_agent_v2, r2.new_messages(), source="user")


Yes, you can join the course late and still receive a certificate as long as you complete the peer-reviewed capstone projects on time. You do not need to complete all homework assignments if you join late. Completion of the capstone projects is essential for certification.

For further details on obtaining your certificate and ensuring your name is correctly displayed, you can check the course profile after grading is completed. Detailed instructions for generating your certificate can be found in the [certificates documentation](https://github.com/DataTalksClub/data-engineering-zoomcamp/blob/main/certificates.md).

For more information, you can refer to the following sources:

- [Do I need to do the homeworks to get the certificate?](https://github.com/DataTalksClub/faq/blob/main/faq-main/_questions/data-engineering-zoomcamp/general/014_3774a79c13_certificate-do-i-need-to-do-the-homeworks-to-get-t.md)
- [How do I get my certificate?](https://github.com/DataTalksClub/faq/blob/main/faq-

WindowsPath('logs/faq_agent_v2_20260329_000556_3f4281.json')

## LLM as a judge

Structured checklist via Pydantic **`output_type`**. Judge model: **`openai:gpt-4o-mini`** (swap if you prefer another model).


In [7]:
evaluation_prompt = """
Use this checklist to evaluate the quality of an AI agent's answer (<ANSWER>) to a user question (<QUESTION>).
We also include the entire log (<LOG>) for analysis.

For each item, check if the condition is met.

Checklist:

- instructions_follow: The agent followed the user's instructions (in <INSTRUCTIONS>)
- instructions_avoid: The agent avoided doing things it was told not to do
- answer_relevant: The response directly addresses the user's question
- answer_clear: The answer is clear and correct
- answer_citations: The response includes proper citations or sources when required
- completeness: The response is complete and covers all key aspects of the request
- tool_call_search: Is the search tool invoked?

Output true/false for each check and provide a short explanation for your judgment.
""".strip()


class EvaluationCheck(BaseModel):
    check_name: str
    justification: str
    check_pass: bool


class EvaluationChecklist(BaseModel):
    checklist: list[EvaluationCheck]
    summary: str


eval_agent = Agent(
    "openai:gpt-4o-mini",
    name="eval_agent",
    instructions=evaluation_prompt,
    output_type=EvaluationChecklist,
)

user_prompt_format = """
<INSTRUCTIONS>{instructions}</INSTRUCTIONS>
<QUESTION>{question}</QUESTION>
<ANSWER>{answer}</ANSWER>
<LOG>{log}</LOG>
""".strip()


In [8]:
def load_log_file(log_file):
    log_file = Path(log_file)
    with log_file.open("r", encoding="utf-8") as f_in:
        log_data = json.load(f_in)
    log_data["log_file"] = str(log_file)
    return log_data


def extract_question_answer(messages):
    """Best-effort extraction from dumped message list."""
    question = None
    answer = None
    for m in messages:
        if m.get("kind") != "request":
            continue
        for p in m.get("parts", []):
            if p.get("part_kind") == "user-prompt":
                c = p.get("content")
                question = c if isinstance(c, str) else str(c)
    for m in reversed(messages):
        if m.get("kind") != "response":
            continue
        for p in m.get("parts", []):
            if p.get("part_kind") == "text":
                c = p.get("content")
                answer = c if isinstance(c, str) else str(c)
                break
        if answer is not None:
            break
    return question, answer


In [9]:
def simplify_log_messages(messages):
    log_simplified = []
    for m in messages:
        parts = []
        for original_part in m.get("parts", []):
            part = dict(original_part)
            kind = part.get("part_kind")
            if kind == "user-prompt":
                part.pop("timestamp", None)
            if kind == "tool-call":
                part.pop("tool_call_id", None)
            if kind == "tool-return":
                part.pop("tool_call_id", None)
                part.pop("metadata", None)
                part.pop("timestamp", None)
                part["content"] = "RETURN_RESULTS_REDACTED"
            if kind == "text":
                part.pop("id", None)
            parts.append(part)
        log_simplified.append({"kind": m.get("kind"), "parts": parts})
    return log_simplified


def evaluate_log_record_sync(eval_agent, log_record):
    messages = log_record["messages"]
    instructions = log_record["system_prompt"]
    question, answer = extract_question_answer(messages)
    log_simplified = simplify_log_messages(messages)
    log = json.dumps(log_simplified)
    user_prompt = user_prompt_format.format(
        instructions=instructions,
        question=question,
        answer=answer,
        log=log,
    )
    result = run_agent_sync(eval_agent, user_prompt)
    return result.output


In [10]:
# Pick the newest faq_agent_v2 log, or create one above first
candidates = sorted(LOG_DIR.glob("faq_agent_v2_*.json"))
if not candidates:
    raise FileNotFoundError("No faq_agent_v2 logs in ./logs — run the v2 cell first.")
log_record = load_log_file(candidates[-1])
eval_out = evaluate_log_record_sync(eval_agent, log_record)
print(eval_out.summary)
for c in eval_out.checklist:
    print(c)


The agent's response effectively follows the provided instructions, is relevant, clear, complete, and includes proper citations.
check_name='instructions_follow' justification='The agent followed the instructions to find relevant information from course materials regarding late registration and certificate eligibility.' check_pass=True
check_name='instructions_avoid' justification='The agent did not contradict any of the specific instructions provided.' check_pass=True
check_name='answer_relevant' justification="The response directly addresses the user's question about joining late and obtaining a certificate." check_pass=True
check_name='answer_clear' justification='The answer is clear in stating the conditions under which a late joiner can receive a certificate.' check_pass=True
check_name='answer_citations' justification='The answer includes proper citations of relevant sources from the course materials.' check_pass=True
check_name='completeness' justification='The response thorough

## Synthetic questions

Sample FAQ rows → LLM writes **natural questions** → run **faq_agent_v2** → log with `source='ai-generated'`.


In [11]:
question_generation_prompt = """
You are helping to create test questions for an AI agent that answers questions about a data engineering course.

Based on the provided FAQ content, generate realistic questions that students might ask.

The questions should:
- Be natural and varied in style
- Range from simple to complex
- Include both specific technical questions and general course questions

Generate one question for each record.
""".strip()


class QuestionsList(BaseModel):
    questions: list[str]


question_generator = Agent(
    "openai:gpt-4o-mini",
    name="question_generator",
    instructions=question_generation_prompt,
    output_type=QuestionsList,
)

sample = random.sample(de_dtc_faq, min(10, len(de_dtc_faq)))
prompt_docs = [d["content"] for d in sample]
prompt = json.dumps(prompt_docs)
qg = run_agent_sync(question_generator, prompt)
questions = qg.output.questions
len(questions), questions[:3]


(10,
 ["I'm encountering an OperationalError with PostgreSQL stating that the database 'ny_taxi' does not exist. What steps should I follow to fix this?",
  'I received an error message indicating that my PostgreSQL server is closing connections unexpectedly. How can I solve this issue?',
  'Can I submit jobs to Dataproc from my local machine? What steps do I need to take to set it up?'])

In [12]:
# Generate ai-generated logs (costs API calls)
for q in tqdm(questions):
    res = run_agent_sync(faq_agent_v2, q)
    log_interaction_to_file(faq_agent_v2, res.new_messages(), source="ai-generated")


  0%|          | 0/10 [00:00<?, ?it/s]

## Aggregate metrics (Pandas)


In [13]:
eval_set = []
for log_file in LOG_DIR.glob("*.json"):
    if "faq_agent_v2" not in log_file.name:
        continue
    rec = load_log_file(log_file)
    if rec.get("source") != "ai-generated":
        continue
    eval_set.append(rec)

eval_results = []
for rec in tqdm(eval_set):
    eval_results.append((rec, evaluate_log_record_sync(eval_agent, rec)))

rows = []
for log_record, eval_result in eval_results:
    messages = log_record["messages"]
    q, a = extract_question_answer(messages)
    row = {
        "file": Path(log_record["log_file"]).name,
        "question": q,
        "answer": a,
    }
    checks = {c.check_name: c.check_pass for c in eval_result.checklist}
    row.update(checks)
    rows.append(row)

df_evals = pd.DataFrame(rows)
df_evals.head()


  0%|          | 0/10 [00:00<?, ?it/s]

,file,question,answer,instructions_follow,instructions_avoid,answer_relevant,answer_clear,answer_citations,completeness,tool_call_search
0,faq_agent_v2_20260329_000615_9f190a.json,I'm encountering an OperationalError with Post...,To resolve the OperationalError indicating tha...,True,True,True,True,True,True,True
1,faq_agent_v2_20260329_000625_1ce512.json,I received an error message indicating that my...,The error message indicating that your Postgre...,True,True,True,True,True,True,True
2,faq_agent_v2_20260329_000633_0c9282.json,Can I submit jobs to Dataproc from my local ma...,"Yes, you can submit jobs to Google Cloud Datap...",True,True,True,True,True,True,True
3,faq_agent_v2_20260329_000643_ec6882.json,What are the benefits of using GitHub Codespac...,Using GitHub Codespaces for your data engineer...,True,True,True,True,True,True,True
4,faq_agent_v2_20260329_000648_3c6bd0.json,How do I read a Gzip compressed CSV file in Pa...,"To read a Gzip compressed CSV file in Pandas, ...",True,True,True,True,True,True,True


In [14]:
# Pass-rate per check (boolean columns)
if not df_evals.empty:
    print(df_evals.mean(numeric_only=True))
else:
    print("No ai-generated logs yet — run the generation loop first.")


instructions_follow    1.0
instructions_avoid     1.0
answer_relevant        1.0
answer_clear           1.0
answer_citations       0.9
completeness           1.0
tool_call_search       1.0
dtype: float64


## Search quality (optional)

For **retrieval** itself, use IR metrics (hit rate, MRR, precision/recall) with labeled `(query, expected_filenames)` pairs — unit-test `text_search` separately from the agent.


In [15]:
def evaluate_search_quality(search_function, test_queries):
    results = []
    for query, expected_docs in test_queries:
        search_results = search_function(query)
        relevant_found = any(doc.get("filename") in expected_docs for doc in search_results)
        mrr = 0.0
        for i, doc in enumerate(search_results):
            if doc.get("filename") in expected_docs:
                mrr = 1.0 / (i + 1)
                break
        results.append({"query": query, "hit": relevant_found, "mrr": mrr})
    return results


# Example (empty labels): fill expected filenames from your FAQ paths
# evaluate_search_quality(lambda q: text_search(q), [("kafka python", {"faq-main/..."})])


## Homework

- Log **≥10** interactions for your agent.  
- Run **LLM-as-judge** on a fixed eval set; compare **prompt / search** variants.  
- Post what you measured.

**Next:** UI & deployment.

[Course](https://alexeygrigorev.com/aihero/) · [Slack `#course-ai-hero`](https://datatalks.club/)
